# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SehrishEjaz1/Flyrank_ML_Intern/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os, subprocess

if not os.path.exists("Flyrank_ML_Intern"):
    subprocess.run(["git", "clone", "https://github.com/SehrishEjaz1/Flyrank_ML_Intern.git"])

os.chdir("Flyrank_ML_Intern")
print("Current dir:", os.getcwd())

Current dir: /content/Flyrank_ML_Intern/Flyrank_ML_Intern


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Lane: Refresh / Content Opportunity Scoring
Task: Binary classification — is this page declining? (yes=1 / no=0)

Method: Logistic Regression first, then Random Forest.

Why Logistic Regression:
- Simple and readable
- Gives probability scores needed for Precision@K ranking
- Coefficients visible — easy leakage check

Why Random Forest:
- Captures combinations: old + low CTR + low impressions together
- Permutation importance lets us verify what it leans on
- More robust to outliers

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["avg_position"] = df["avg_position"].replace(0, np.nan)

print("Shape:", df.shape)
print("Base rate:", round(df["is_declining_label"].mean(), 3))

Shape: (30000, 45)
Base rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split type: Grouped by client_id

Why client-grouped:
- Same client's pages share content strategy and history
- Random split would leak client patterns into test set
- Grouped split = honest estimate on NEW clients

80% clients → train | 20% clients → test

In [8]:
np.random.seed(SEED)
clients = df["client_id"].unique()
np.random.shuffle(clients)

split = int(len(clients) * 0.8)
train_clients = clients[:split]
test_clients  = clients[split:]

train = df[df["client_id"].isin(train_clients)].copy()
test  = df[df["client_id"].isin(test_clients)].copy()

print(f"Train: {len(train)} rows, {len(train_clients)} clients")
print(f"Test:  {len(test)} rows, {len(test_clients)} clients")
print(f"Train base rate: {train['is_declining_label'].mean():.3f}")
print(f"Test base rate:  {test['is_declining_label'].mean():.3f}")

# Features — no leakage
train["has_wordcount"] = train["word_count"].notna().astype(int)
test["has_wordcount"]  = test["word_count"].notna().astype(int)
train["word_count"] = train["word_count"].fillna(0)
test["word_count"]  = test["word_count"].fillna(0)

features = ["days_since_last_update", "impressions_90d", "ctr",
            "avg_position", "content_age_days", "word_count", "has_wordcount"]

X_train = train[features].fillna(0)
y_train = train["is_declining_label"]
X_test  = test[features].fillna(0)
y_test  = test["is_declining_label"]

print(f"\nFeatures: {features}")

Train: 22389 rows, 25 clients
Test:  7611 rows, 7 clients
Train base rate: 0.546
Test base rate:  0.531

Features: ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'word_count', 'has_wordcount']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same metric (Precision@50) as Week-4 baseline.
Baseline score recreated here from the same test data.

In [9]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk  = np.asarray(labels)[order[:k]]
    return topk.mean()

# Week-4 baseline on same test split
ctr_median = train["ctr"].median()
stale   = (test["days_since_last_update"] >= 180).astype(int)
visible = (test["impressions_90d"] >= 500).astype(int)
low_ctr = (test["ctr"] < ctr_median).astype(int)
baseline_scores = stale * visible * (1 + low_ctr) * test["impressions_90d"]
p_baseline = precision_at_k(baseline_scores, y_test)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_test)[:, 1]
p_lr = precision_at_k(lr_scores, y_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6,
                             random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
p_rf = precision_at_k(rf_scores, y_test)

base_rate = y_test.mean()

# Comparison table
results = pd.DataFrame({
    "Method": ["Base rate (random)", "Week-4 Rule Baseline",
                "Logistic Regression", "Random Forest"],
    "Precision@50": [base_rate, p_baseline, p_lr, p_rf]
})
print(results.to_string(index=False))

# Feature importances
imp_df = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
print("\nRandom Forest Feature Importances:")
print(imp_df.to_string(index=False))

              Method  Precision@50
  Base rate (random)      0.531468
Week-4 Rule Baseline      0.500000
 Logistic Regression      0.280000
       Random Forest      0.720000

Random Forest Feature Importances:
               feature  importance
       impressions_90d    0.357348
          avg_position    0.274611
      content_age_days    0.156798
            word_count    0.082518
days_since_last_update    0.052829
                   ctr    0.048779
         has_wordcount    0.027116


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What the model leans on:
- days_since_last_update: older pages more likely declining (directional)
- impressions_90d: high traffic pages more visible to label (observed)
- ctr: low CTR correlates with declining engagement (measured)

Where it gets it wrong:
- False positives: high impressions pages flagged but actually stable
- False negatives: new pages with low CTR missed — model thinks fresh = safe

These errors are expected — label is a proxy, not ground truth.
Output is decision-support, not a final verdict.

In [10]:
test_copy = test.copy()
test_copy["rf_score"]  = rf_scores
test_copy["predicted"] = (rf_scores >= 0.5).astype(int)

show_cols = ["days_since_last_update", "impressions_90d",
             "ctr", "rf_score", "is_declining_label"]

fp = test_copy[
    (test_copy["predicted"] == 1) &
    (test_copy["is_declining_label"] == 0)
].sort_values("rf_score", ascending=False).head(3)

fn = test_copy[
    (test_copy["predicted"] == 0) &
    (test_copy["is_declining_label"] == 1)
].sort_values("rf_score").head(3)

print("=== False Positives (flagged but not declining) ===")
print(fp[show_cols].to_string())
print("\n=== False Negatives (missed — actually declining) ===")
print(fn[show_cols].to_string())

=== False Positives (flagged but not declining) ===
       days_since_last_update  impressions_90d   ctr  rf_score  is_declining_label
13751                      20            71283  0.08  0.781068                   0
19589                     104            10539  0.03  0.777222                   0
13839                     104             2474  0.08  0.776893                   0

=== False Negatives (missed — actually declining) ===
       days_since_last_update  impressions_90d    ctr  rf_score  is_declining_label
3879                       20                3   0.00  0.036995                   1
22991                      20                3  33.33  0.147982                   1
1371                       20                2   0.00  0.189866                   1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.